# Занятие 3. Фильтрация данных и обработка пропусков

> **Цель занятия:** научиться отбирать нужные строки из таблицы по условиям
> и справляться с «дырками» в данных.

**Что будет:**
1. **Булево маскирование** — как отбирать строки по условию;
2. **Сложные условия** — объединяем через **И (`&`)**, **ИЛИ (`|`)**, **НЕ (`~`)**;
3. **`.query()`** — более читаемый способ записи условий;
4. **`.copy()`** — зачем копировать таблицу перед изменениями;
5. **Пропущенные значения** — как их находить (`.isna()`), удалять (`.dropna()`)
   и заполнять (`.fillna()`).

Все примеры разбираем на **датасете** из прошлого занятия — таблице с результатами ЕГЭ.

In [1]:
import numpy as np
import pandas as pd

## Немного об удобстве: настройки отображения Pandas

По умолчанию Pandas старается **не перегружать экран**: если столбцов много,
часть из них он прячет и заменяет многоточием `...`. То же самое со строками и
широкими текстами — они обрезаются. Для маленьких табличек это удобно, но когда
мы работаем с реальными данными (у нас **16 столбцов**), хочется **видеть
всё сразу**.

Управлять этим помогает функция **`pd.set_option()`**. Она меняет глобальные
настройки отображения — как таблица будет **печататься** на экран.
Сами данные при этом не меняются, меняется только их «вид».

**Самые полезные настройки:**

| Настройка | Что делает | Пример |
|---|---|---|
| `display.max_columns` | сколько столбцов показывать целиком | `pd.set_option('display.max_columns', 20)` |
| `display.max_rows` | сколько строк показывать | `pd.set_option('display.max_rows', 100)` |
| `display.width` | ширина вывода в символах | `pd.set_option('display.width', 200)` |
| `display.max_colwidth` | максимальная ширина одной ячейки | `pd.set_option('display.max_colwidth', 100)` |
| `display.precision` | сколько знаков после запятой у чисел | `pd.set_option('display.precision', 2)` |

**Как это работает:**
- значение `20` — «показывай до 20 столбцов, не прячь»;
- значение `None` — «не ограничивай вообще, показывай **всё**»;
- вернуть настройку по умолчанию можно через `pd.reset_option('display.max_columns')`.

> **Совет:** в начале ноутбука полезно один раз задать удобные настройки —
> дальше все `df.head()` и подобные вызовы будут показывать данные так, как вам
> нравится, без сюрпризов с многоточиями.

Поставим `max_columns = 20`, чтобы все 16 столбцов таблицы влезли на экран целиком.

In [2]:
pd.set_option('display.max_columns', 20)      # показывать до 20 столбцов
pd.set_option('display.width', 200)           # шире область вывода
pd.set_option('display.precision', 2)         # 2 знака после запятой у чисел

## Загружаем данные про ЕГЭ

Файл `ege_students.csv`. В нём **1000 учеников** и **16 столбцов**:
имя, пол, возраст, город, школа, любимый предмет, часы подготовки, репетитор, пробники,
пропуски уроков, мотивация, часы сна, средний балл года, балл ЕГЭ и признак «сдал».

In [3]:
df = pd.read_csv('data/ege_students.csv', sep=';')
print('Размер таблицы:', df.shape)
df.head()

Размер таблицы: (1000, 16)


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


In [4]:
df.dtypes

ученик_id                      int64
имя                           object
пол                           object
возраст                        int64
город                         object
тип_школы                     object
любимый_предмет               object
часов_подготовки_в_неделю    float64
репетитор                     object
кол_во_пробников               int64
пропусков_уроков               int64
мотивация_1_10               float64
часов_сна                    float64
средний_балл_года            float64
балл_егэ                       int64
сдал                          object
dtype: object

---
## Часть 1. Булево маскирование — отбираем строки по условию

Часто нужно выбрать не всю таблицу, а только её **часть**: «только девочки»,
«только те, кто сдал», «только с баллом выше 80». Для этого в Pandas есть
специальный приём — **булево маскирование** (или «логическое индексирование»).

**Идея простая, в два шага:**
1. Пишем условие про столбец — получаем **маску**: колонку из значений `True` / `False`,
   по одному на каждую строку таблицы.
2. Подставляем эту маску в квадратные скобки `df[...]` — Pandas **оставит только те строки,
   где стоит `True`**.

Сначала посмотрим, как выглядит маска.

In [5]:
# условие: балл ЕГЭ выше 80
mask = df['балл_егэ'] > 80
mask.head(10)

0    False
1    False
2    False
3    False
4    False
5     True
6     True
7    False
8    False
9    False
Name: балл_егэ, dtype: bool

**Что мы видим:** для каждой строки — `True` или `False`.
`True` там, где балл ЕГЭ действительно больше 80, и `False` там, где нет.

Теперь применим маску к таблице.

In [6]:
high_score = df[df['балл_егэ'] > 80]
print('Всего строк:', len(df))
print('С баллом > 80:', len(high_score))
high_score.head()

Всего строк: 1000
С баллом > 80: 256


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да
6,7,Роман,М,17,Самара,Лицей,Физика,16.6,нет,8,2,7.0,6.8,3.9,90,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да
16,17,Полина,Ж,17,Казань,Лицей,Химия,9.2,да,9,3,4.0,7.3,3.4,83,да
18,19,София,Ж,17,Ростов-на-Дону,Обычная,Информатика,8.7,нет,6,2,5.0,8.2,3.1,81,да


**Обратите внимание:** мы написали `df[df['балл_егэ'] > 80]` — то есть
**внутри квадратных скобок** оказалась не строка и не число, а **условие**.
Именно так и работает булево маскирование.

### Простые примеры условий

Так же можно фильтровать и по текстовым столбцам — через `==` (равно):

In [7]:
# только ученики из Москвы
df[df['город'] == 'Москва'].head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
11,12,Ева,Ж,18,Москва,Обычная,Русский язык,10.0,нет,8,7,7.0,5.4,2.1,58,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да
21,22,Дмитрий,М,17,Москва,Обычная,Обществознание,3.9,нет,9,3,2.0,7.9,2.3,49,да
31,32,Максим,М,17,Москва,Обычная,Биология,6.3,да,7,3,6.0,7.5,2.2,70,да


In [8]:
# только те, кто НЕ сдал
df[df['сдал'] == 'нет'].head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
46,47,Иван,М,16,Санкт-Петербург,Обычная,Обществознание,3.0,да,8,9,4.0,6.6,3.0,40,нет
59,60,Виктория,Ж,18,Самара,Гимназия,Биология,3.7,нет,5,10,5.0,7.1,2.0,26,нет
145,146,Ксения,Ж,17,Новосибирск,Лицей,Обществознание,8.2,нет,5,7,3.0,6.2,2.6,38,нет
174,175,София,Ж,17,Казань,Лицей,История,0.0,да,5,11,5.0,8.4,2.0,25,нет
175,176,Артём,М,18,Самара,Обычная,Информатика,12.8,да,5,10,2.0,6.7,2.0,39,нет


### Что за операторы можно использовать?

| Оператор | Смысл | Пример |
|---|---|---|
| `==` | равно | `df['пол'] == 'Ж'` |
| `!=` | не равно | `df['город'] != 'Москва'` |
| `>` `>=` | больше / больше-равно | `df['балл_егэ'] >= 80` |
| `<` `<=` | меньше / меньше-равно | `df['возраст'] < 18` |
| `.isin([...])` | значение из списка | `df['город'].isin(['Москва', 'Казань'])` |

Про `.isin()` отдельно: удобно, когда нужно проверить, попадает ли значение в **список**
допустимых. Иначе пришлось бы писать длинную цепочку `или`.

In [9]:
# ученики из двух столиц
cities = ['Москва', 'Санкт-Петербург']
df[df['город'].isin(cities)].head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
8,9,Тимофей,М,16,Санкт-Петербург,Лицей,Биология,8.0,да,4,5,8.0,8.0,2.2,57,да
10,11,Алиса,Ж,16,Санкт-Петербург,Обычная,Биология,7.6,нет,6,4,5.0,NaN,2.9,64,да
11,12,Ева,Ж,18,Москва,Обычная,Русский язык,10.0,нет,8,7,7.0,5.4,2.1,58,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да


---
## Часть 2. Сложные условия: И, ИЛИ, НЕ

Одного условия часто мало. Например: «девочки **И** сдали», «Москва **ИЛИ** Питер»,
«**НЕ** из Москвы». Такие сложные условия собирают из простых через **логические операторы**.

**В Pandas свои операторы** — не такие, как в обычном Python:

| Что означает | Обычный Python | В Pandas (с масками) |
|---|---|---|
| **И** — оба условия верны | `and` | `&` |
| **ИЛИ** — хотя бы одно верно | `or` | `\|` |
| **НЕ** — отрицание | `not` | `~` |

> **Важное правило:** каждое условие **оборачиваем в круглые скобки**.
> Без скобок Pandas запутается в порядке операций и выдаст ошибку.

### Оператор И (`&`) — оба условия одновременно

In [10]:
# девочки И сдали
mask = (df['пол'] == 'Ж') & (df['сдал'] == 'да')
df[mask].head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да
7,8,Дарья,Ж,17,Нижний Новгород,Обычная,Биология,0.0,нет,7,12,4.0,8.6,2.5,48,да
10,11,Алиса,Ж,16,Санкт-Петербург,Обычная,Биология,7.6,нет,6,4,5.0,NaN,2.9,64,да


In [11]:
# балл ЕГЭ выше 80 И больше 10 часов подготовки в неделю
mask = (df['балл_егэ'] > 80) & (df['часов_подготовки_в_неделю'] > 10)
print('Подходят:', mask.sum(), 'учеников')
df[mask].head()

Подходят: 163 учеников


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
6,7,Роман,М,17,Самара,Лицей,Физика,16.6,нет,8,2,7.0,6.8,3.9,90,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да
23,24,Ева,Ж,17,Новосибирск,Обычная,Биология,12.2,да,8,2,6.0,7.5,3.1,83,да
24,25,Ева,Ж,17,Новосибирск,Обычная,Обществознание,14.6,да,6,2,7.0,10.1,3.9,100,да
25,26,София,Ж,17,Санкт-Петербург,Обычная,Информатика,14.3,нет,8,1,7.0,8.3,4.3,88,да


### Оператор ИЛИ (`|`) — хотя бы одно из условий

In [12]:
# из Москвы ИЛИ из Санкт-Петербурга
mask = (df['город'] == 'Москва') | (df['город'] == 'Санкт-Петербург')
print('Из двух столиц:', mask.sum())
df[mask].head()

Из двух столиц: 254


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
8,9,Тимофей,М,16,Санкт-Петербург,Лицей,Биология,8.0,да,4,5,8.0,8.0,2.2,57,да
10,11,Алиса,Ж,16,Санкт-Петербург,Обычная,Биология,7.6,нет,6,4,5.0,NaN,2.9,64,да
11,12,Ева,Ж,18,Москва,Обычная,Русский язык,10.0,нет,8,7,7.0,5.4,2.1,58,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да


### Оператор НЕ (`~`) — переворачивает True в False

In [13]:
# НЕ из Москвы
mask = ~(df['город'] == 'Москва')
print('Не из Москвы:', mask.sum())
df[mask].head()

Не из Москвы: 882


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да


### Комбинируем несколько условий

Условий может быть сколько угодно — главное каждое **в скобках** и правильный оператор между ними.

In [14]:
# девочки-лицеистки, которые сдали на 70+
mask = (df['пол'] == 'Ж') & (df['тип_школы'] == 'Лицей') & (df['балл_егэ'] >= 70)
print('Подходят:', mask.sum())
df[mask].head()

Подходят: 59


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
16,17,Полина,Ж,17,Казань,Лицей,Химия,9.2,да,9,3,4.0,7.3,3.4,83,да
42,43,Полина,Ж,17,Нижний Новгород,Лицей,Русский язык,11.6,нет,7,6,10.0,7.3,3.7,75,да
50,51,Дарья,Ж,17,Санкт-Петербург,Лицей,Информатика,4.5,да,3,2,6.0,NaN,2.7,72,да
179,180,София,Ж,17,Самара,Лицей,История,5.6,да,9,2,6.0,5.8,3.4,82,да
211,212,Арина,Ж,17,Новосибирск,Лицей,Химия,11.5,да,7,8,6.0,6.1,3.0,75,да


> **Частая ошибка:** написать `and` вместо `&` или забыть скобки.
> Тогда Python выдаёт ошибку про «truth value of a Series is ambiguous».
> Если увидите её — проверьте: везде ли `&` `|` `~` и все ли условия в круглых скобках.

---
## Часть 3. Метод `.query()` — то же самое, но читаемее

Когда условий много, запись со скобками и амперсандами становится громоздкой.
У Pandas есть **альтернатива** — метод **`.query()`**. В нём условие пишут **строкой**,
почти как обычным языком.

Сравните две записи одного и того же:

In [15]:
# через булеву маску
df[(df['пол'] == 'Ж') & (df['балл_егэ'] >= 70)].head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да
13,14,Виктория,Ж,18,Ростов-на-Дону,Обычная,Математика,10.1,нет,12,6,7.0,8.5,3.4,78,да
15,16,Ева,Ж,17,Новосибирск,Гимназия,Английский язык,8.8,нет,6,3,5.0,NaN,3.2,70,да


In [16]:
# то же самое через .query()
df.query('пол == "Ж" and балл_егэ >= 70').head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да
13,14,Виктория,Ж,18,Ростов-на-Дону,Обычная,Математика,10.1,нет,12,6,7.0,8.5,3.4,78,да
15,16,Ева,Ж,17,Новосибирск,Гимназия,Английский язык,8.8,нет,6,3,5.0,NaN,3.2,70,да


**Что удобно в `.query()`:**
- пишем **`and` / `or` / `not`** — как в обычном Python;
- **не нужны** ни квадратные скобки `df[...]`, ни лишние `df['...']` вокруг имён столбцов;
- условие читается почти как фраза на языке.

**Синтаксис:**
- имена столбцов пишем **прямо** (без кавычек и без `df[...]`);
- **строковые значения** — в кавычках: `'да'` или `"да"`;
- числа — как есть.

### Ещё примеры

In [17]:
# балл ЕГЭ выше 80 и больше 10 часов подготовки
df.query('балл_егэ > 80 and часов_подготовки_в_неделю > 10').head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
6,7,Роман,М,17,Самара,Лицей,Физика,16.6,нет,8,2,7.0,6.8,3.9,90,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да
23,24,Ева,Ж,17,Новосибирск,Обычная,Биология,12.2,да,8,2,6.0,7.5,3.1,83,да
24,25,Ева,Ж,17,Новосибирск,Обычная,Обществознание,14.6,да,6,2,7.0,10.1,3.9,100,да
25,26,София,Ж,17,Санкт-Петербург,Обычная,Информатика,14.3,нет,8,1,7.0,8.3,4.3,88,да


In [18]:
# из Москвы или Питера
df.query('город == "Москва" or город == "Санкт-Петербург"').head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
8,9,Тимофей,М,16,Санкт-Петербург,Лицей,Биология,8.0,да,4,5,8.0,8.0,2.2,57,да
10,11,Алиса,Ж,16,Санкт-Петербург,Обычная,Биология,7.6,нет,6,4,5.0,NaN,2.9,64,да
11,12,Ева,Ж,18,Москва,Обычная,Русский язык,10.0,нет,8,7,7.0,5.4,2.1,58,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да


In [19]:
# то же через in — короче
df.query('город in ["Москва", "Санкт-Петербург"]').head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
8,9,Тимофей,М,16,Санкт-Петербург,Лицей,Биология,8.0,да,4,5,8.0,8.0,2.2,57,да
10,11,Алиса,Ж,16,Санкт-Петербург,Обычная,Биология,7.6,нет,6,4,5.0,NaN,2.9,64,да
11,12,Ева,Ж,18,Москва,Обычная,Русский язык,10.0,нет,8,7,7.0,5.4,2.1,58,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да


In [20]:
# НЕ из Москвы
df.query('город != "Москва"').head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да


### Когда какой способ удобнее?

| Способ | Когда удобнее |
|---|---|
| **Булева маска** `df[...]` | если условие **простое** (одно-два), или маску нужно **сохранить в переменную** и использовать много раз |
| **`.query()`** | если условий **много**, и хочется чтобы запись читалась как обычный текст |

**Оба способа полностью равнозначны** — они возвращают одну и ту же таблицу. Выбирайте тот,
который вам кажется яснее в конкретной ситуации.

---
## Часть 4. `.copy()` — зачем копировать таблицу

Когда мы отбираем часть строк — `df_high = df[df['балл_егэ'] > 80]` — Pandas часто
возвращает **не самостоятельную новую таблицу, а «взгляд»** (view) на кусок исходной.

Если потом попробовать что-то поменять в этом кусочке, Pandas ругается предупреждением
`SettingWithCopyWarning` — потому что непонятно, куда именно применять изменения:
в кусочек или в исходную таблицу.

**Решение простое:** если вы собираетесь **менять** отфильтрованные данные — сразу вызывайте
`.copy()`. Это делает **самостоятельную копию**, с которой можно работать безопасно.

In [21]:
# делаем самостоятельную копию — с ней можно работать спокойно
high_score = df[df['балл_егэ'] > 80].copy()
high_score['отлично'] = 'да'   # добавили столбец в копию
high_score.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,отлично
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да,да
6,7,Роман,М,17,Самара,Лицей,Физика,16.6,нет,8,2,7.0,6.8,3.9,90,да,да
12,13,Алиса,Ж,17,Москва,Гимназия,История,11.3,нет,11,6,10.0,7.4,3.2,89,да,да
16,17,Полина,Ж,17,Казань,Лицей,Химия,9.2,да,9,3,4.0,7.3,3.4,83,да,да
18,19,София,Ж,17,Ростов-на-Дону,Обычная,Информатика,8.7,нет,6,2,5.0,8.2,3.1,81,да,да


In [22]:
# в исходной таблице никакого столбца 'отлично' нет — copy сработал
print('Столбцы df:', list(df.columns))

Столбцы df: ['ученик_id', 'имя', 'пол', 'возраст', 'город', 'тип_школы', 'любимый_предмет', 'часов_подготовки_в_неделю', 'репетитор', 'кол_во_пробников', 'пропусков_уроков', 'мотивация_1_10', 'часов_сна', 'средний_балл_года', 'балл_егэ', 'сдал']


**Правило простое:**
> Если после фильтрации будете **что-то менять** — вызывайте `.copy()`.
> Если только смотрите или считаете — можно без него.

---
## Часть 5. Пропущенные значения (NaN)

В реальных данных часто чего-то **не хватает**: ученик забыл заполнить анкету, датчик
сломался, поле оставили пустым. Такие «дырки» в Pandas обозначаются специальным значением
**`NaN`** (от англ. *Not a Number* — «не число»).

Работать с NaN нужно осторожно: обычные вычисления с ними дают снова NaN
(`5 + NaN = NaN`), а сравнения возвращают `False`.

### Шаг 1. Находим пропуски — `.isna()`

Метод **`.isna()`** возвращает такую же таблицу из `True` / `False`: `True` — где пропуск.
Ещё есть `.notna()` — наоборот, `True` где значение есть.

In [23]:
# сколько пропусков в каждом столбце
df.isna().sum()

ученик_id                     0
имя                           0
пол                           0
возраст                       0
город                         0
тип_школы                     0
любимый_предмет               0
часов_подготовки_в_неделю     0
репетитор                     0
кол_во_пробников              0
пропусков_уроков              0
мотивация_1_10               25
часов_сна                    40
средний_балл_года             0
балл_егэ                      0
сдал                          0
dtype: int64

Видим: пропуски есть в столбцах **`мотивация_1_10`** и **`часов_сна`**. В остальных всё заполнено.
Разберёмся с ними.

In [24]:
# посмотрим на строки, где нет мотивации
df[df['мотивация_1_10'].isna()]

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
54,55,Дмитрий,М,16,Новосибирск,Обычная,Биология,5.8,нет,8,7,NaN,5.7,3.1,69,да
113,114,Артём,М,18,Москва,Обычная,История,10.4,нет,12,5,NaN,NaN,3.4,69,да
128,129,Дарья,Ж,17,Екатеринбург,Обычная,Биология,14.8,нет,4,5,NaN,7.0,3.6,95,да
217,218,Тимофей,М,17,Екатеринбург,Обычная,Русский язык,1.0,да,6,4,NaN,6.6,2.7,52,да
219,220,Ксения,Ж,17,Нижний Новгород,Обычная,Английский язык,6.6,да,4,6,NaN,8.1,2.4,68,да
229,230,Артём,М,17,Ростов-на-Дону,Гимназия,История,3.6,да,11,12,NaN,6.9,2.2,55,да
232,233,Матвей,М,17,Москва,Обычная,Химия,1.6,нет,6,7,NaN,4.6,2.0,46,да
300,301,Тимофей,М,17,Москва,Обычная,История,9.1,да,15,3,NaN,5.8,3.5,99,да
319,320,Полина,Ж,17,Самара,Обычная,Русский язык,6.8,нет,2,8,NaN,6.3,2.2,48,да
320,321,Дмитрий,М,16,Самара,Лицей,Химия,17.0,да,6,6,NaN,7.5,3.3,71,да


### Шаг 2. Удаляем строки с пропусками — `.dropna()`

Самый простой способ — **выкинуть** строки, в которых есть пропуски. Метод
**`.dropna()`** удаляет все строки, где хотя бы в одном столбце стоит `NaN`.

In [25]:
df_clean = df.dropna()
print('Было строк:', len(df))
print('Осталось после dropna:', len(df_clean))
print('Удалили:', len(df) - len(df_clean), 'строк с пропусками')

Было строк: 1000
Осталось после dropna: 936
Удалили: 64 строк с пропусками


Можно удалять пропуски **только по одному столбцу**, если остальные не так важны:

In [26]:
# выбрасываем только те строки, где нет часов сна
df_sleep = df.dropna(subset=['часов_сна'])
print('Осталось:', len(df_sleep))

Осталось: 960


> **Осторожно:** `.dropna()` может выкинуть очень много данных, если пропусков много.
> В нашем случае — всего ~65 строк из 1000, потеря небольшая. Но если пропусков 30–50%,
> лучше их **заполнить**, а не удалять.

### Шаг 3. Заполняем пропуски — `.fillna()`

Метод **`.fillna(значение)`** ставит на место каждого `NaN` то, что мы укажем.
Часто используют:
- **среднее** значение по столбцу — для числовых данных;
- **медиану** — если есть выбросы;
- **`0`** или специальное значение — если это осмысленно;
- **самое частое значение** — для категорий.

In [27]:
# заполним пропуски в часах сна средним значением
avg_sleep = df['часов_сна'].mean()
print('Среднее число часов сна:', round(avg_sleep, 2))

df_filled = df.copy()
df_filled['часов_сна'] = df_filled['часов_сна'].fillna(avg_sleep)
print('Пропусков в часах сна теперь:', df_filled['часов_сна'].isna().sum())

Среднее число часов сна: 6.98
Пропусков в часах сна теперь: 0


In [28]:
# мотивацию заполним медианой (это оценка от 1 до 10 — целое число)
med_mot = df['мотивация_1_10'].median()
print('Медиана мотивации:', med_mot)

df_filled['мотивация_1_10'] = df_filled['мотивация_1_10'].fillna(med_mot)
print('Пропусков теперь:', df_filled['мотивация_1_10'].isna().sum())

Медиана мотивации: 6.0
Пропусков теперь: 0


In [29]:
# проверим: пропусков больше нет
df_filled.isna().sum()

ученик_id                    0
имя                          0
пол                          0
возраст                      0
город                        0
тип_школы                    0
любимый_предмет              0
часов_подготовки_в_неделю    0
репетитор                    0
кол_во_пробников             0
пропусков_уроков             0
мотивация_1_10               0
часов_сна                    0
средний_балл_года            0
балл_егэ                     0
сдал                         0
dtype: int64

### Что выбрать: удалять или заполнять?

| Ситуация | Что делать |
|---|---|
| Пропусков **очень мало** (< 5%) | Проще **удалить** через `.dropna()` |
| Пропусков много, столбец важный | **Заполнить** через `.fillna()` (среднее / медиана / мода) |
| Столбец **почти пустой** | Удалить весь столбец |
| Пропуск — это **осмысленный «ноль»** | Заполнить нулём: `.fillna(0)` |

**Главное правило:** прежде чем что-то делать с пропусками — **посмотрите, где они и сколько их**.
Обычно это первый шаг любого анализа.

---
## Итог занятия

Сегодня научились:
- **отбирать строки** по условию через **булеву маску** `df[df['столбец'] > ...]`;
- собирать **сложные условия** через `&` (И), `|` (ИЛИ), `~` (НЕ) — не забывая скобки;
- писать те же условия **читаемо** через **`.query()`**;
- копировать таблицу через **`.copy()`**, если собираемся её менять;
- находить пропуски через **`.isna()`**, удалять их через **`.dropna()`**
  и заполнять через **`.fillna()`**.

**Главная мысль:** реальные данные почти никогда не бывают идеальными.
Уметь **быстро отобрать нужный кусок** и **аккуратно обработать пропуски** — базовый
навык любого анализа. Без него все дальнейшие расчёты будут неверными.